In [1]:
import os, glob, json, re, uuid, math, hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from datetime import date, datetime, timedelta, timezone
from multiprocessing import Pool
from pathlib import Path
from time import perf_counter
from typing import Any, Iterable, Optional, Annotated
from sentence_transformers import SentenceTransformer
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
)
import pandas as pd
import torch
from embedder import Embedder, PointShardWriter
from datetime import datetime, date, UTC
from tqdm.auto import tqdm
from summorizations import summarize_row, stable_id
from point_builder import build_point_dict


In [2]:
DEFAULT_EMBEDDER = "BAAI/bge-m3"  # 1024-dim
DEFAULT_VECTOR_SIZE = 1024
os.environ["TOKENIZERS_PARALLELISM"] = "false"

emb: Embedder = Embedder(DEFAULT_EMBEDDER)
CACHE_DIR = "./cache/bq_user_daily_metrics"
OUT_DIR   = "/content/points_out"

[Embedder] Using Apple MPS


In [3]:
def safe_read_parquet(path: str) -> pd.DataFrame:
    try:
        # Fast path (works for normal Parquet files)
        return pd.read_parquet(path, engine="pyarrow")
    except Exception as e:
        # Fallback: ignore pandas metadata that includes bad dtypes like "dbdate"
        import pyarrow.parquet as pq

        table = pq.read_table(path)
        df = table.to_pandas(ignore_metadata=True)
        return df


In [ ]:


FILE_GLOB = "*.parquet"  # change to "*.csv" if needed
BATCH_SIZE_EMB = 2048    # tune by GPU/VRAM
SHARD_SIZE = 25_000      # ~25k points per gzip shard

# Discover files
files = sorted(glob.glob(os.path.join(CACHE_DIR, "**", FILE_GLOB), recursive=True))
print("Found", len(files), "files")

writer = PointShardWriter(OUT_DIR, shard_size=SHARD_SIZE, basename="qdrant_points")

seen_ids = set()
manifest = []  # list of {"file","rows_in","rows_out","shards_written"}

total_in = 0
total_out = 0

for path in tqdm(files):
    # Read dataframe
    if path.endswith(".parquet"):
        df = safe_read_parquet(path)
    elif path.endswith(".csv"):
        df = pd.read_csv(path)
    else:
        continue

    if df.empty:
        manifest.append({"file":path, "rows_in":0, "rows_out":0})
        continue

    # Normalize date_dim to ISO string if needed
    if "date_dim" in df.columns:
        try:
            df["date_dim"] = pd.to_datetime(df["date_dim"]).dt.date.astype(str)
        except Exception:
            df["date_dim"] = df["date_dim"].astype(str)

    rows = df.to_dict("records")
    total_in += len(rows)

    # Build summaries
    texts = [summarize_row(r) for r in rows]

    # Embed in chunks
    shard_written_before = writer.shard_idx
    out_count_before = total_out

    for i in range(0, len(rows), BATCH_SIZE_EMB):
        chunk_rows = rows[i:i+BATCH_SIZE_EMB]
        chunk_texts = texts[i:i+BATCH_SIZE_EMB]
        dense = emb.encode(chunk_texts, batch_size=BATCH_SIZE_EMB)

        # Build point dicts (skip duplicates)
        pts = []
        for r, t, v in zip(chunk_rows, chunk_texts, dense):
            pid = stable_id(str(r.get("user_id")), str(r.get("date_dim")))
            if pid in seen_ids:
                continue
            seen_ids.add(pid)
            pts.append(build_point_dict(r, t, v))

        writer.write_many(pts)
        total_out += len(pts)

    shards_written = writer.shard_idx - shard_written_before
    manifest.append({"file": path, "rows_in": len(rows),
                     "rows_out": total_out - out_count_before,
                     "shards_written": shards_written})

writer.close()
print(f"Done. Total rows in: {total_in}, unique points out: {total_out}")

Found 32 files


  0%|          | 0/32 [00:00<?, ?it/s]